# Rosenxt_bot: Send LLM Answers to Telegram

This script uses **LangChain + Ollama** to answer a user question with `llama3.1:latest` and then sends the question and answer to a Telegram **supergroup/channel** via the Telegram Bot API.



In [ ]:
import requests
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

TELEGRAM_TOKEN = "8454954656:AAFwJMW2p83kjMk8GN7xkoMuPTh0iBxXScM"
TELEGRAM_CHAT_ID = "-1003718106218"


def send_to_telegram(text: str) -> None:
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    params = {"chat_id": TELEGRAM_CHAT_ID, "text": text}
    resp = requests.post(url, params=params)
    resp.raise_for_status()


# --- LLM setup ---
llm_model = ChatOllama(model="llama3.1:latest")

template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are Rosenxt_bot.
The user gives you a natural-language sentence about a payment.
Extract:
- price (amount of money, only number)
- date (YYYY/MM/DD; if not explicit, infer from words like 'today')
- purpose (what the payment is for)
- payer (who paid)

Output EXACTLY in this format:

Price: <amount> Toman
date: <YYYY/MM/DD>
For: <short description>
From: <name>

Do not add anything else (no explanations, no extra text).""",
        ),
        (
            "human",
            "User text: {query}"
        ),
    ]
)

chain = template | llm_model


def answer_and_push(user_text: str):
    # 1) Let LLM extract & format
    result = chain.invoke({"query": user_text})
    answer = result.content

    # 2) Send formatted message to Telegram
    send_to_telegram(answer)

    return answer


if __name__ == "__main__":
    user_q = "today, Pouriya paid it for Window of house, 12 Toman"
    answer = answer_and_push(user_q)
    print("Answer: ", answer)


Answer:
 Price: 12 
date: Today
For: House window payment
From: Pouriya
